# County Data Exploratory Analysis

## Purpose
The purpose of this notebook is to profile the parcel dataset received from Kalamazoo County Treasury, identify data quality issues, understand relationships among parcels, owners, taxpayers, and mailing addresses, and gather information needed to design a relational database and future entity-resolution pipeline.

## Importing data and Pandas package

In [1]:
import pandas as pd
import re

# Define a function to both clean and normalize addresses
def normalize_address(series):
    """
    Normalize address strings by:
    - Converting to uppercase
    - Replacing NaN with ""
    - Removing extra whitespace
    - Standardizing common street suffixes and directions
    - Removing spaces before commas
    - Converting placeholder ', ,' addresses to ""
    """

    # Basic cleaning
    series = (
        series.fillna("")
              .astype(str)
              .str.upper()
              .str.replace(r"\s+", " ", regex=True)
              .str.strip()
    )

    # Standardization dictionary
    replacements = {
        r"\bAVENUE\b": "AVE",
        r"\bROAD\b": "RD",
        r"\bSTREET\b": "ST",
        r"\bDRIVE\b": "DR",
        r"\bCOURT\b": "CT",
        r"\bLANE\b": "LN",
        r"\bBOULEVARD\b": "BLVD",
        r"\bPLACE\b": "PL",
        r"\bCIRCLE\b": "CIR",
        r"\bPARKWAY\b": "PKWY",
        r"\bHIGHWAY\b": "HWY",
        r"\bTRAIL\b": "TRL",
        r"\bTERRACE\b": "TER",
        r"\bSQUARE\b": "SQ",
        r"\bAPARTMENT\b": "APT",
        r"\bSUITE\b": "STE",
        r"\bBUILDING\b": "BLDG",

        # Directions
        r"\bNORTH\b": "N",
        r"\bSOUTH\b": "S",
        r"\bEAST\b": "E",
        r"\bWEST\b": "W",
        r"\bNORTHEAST\b": "NE",
        r"\bNORTHWEST\b": "NW",
        r"\bSOUTHEAST\b": "SE",
        r"\bSOUTHWEST\b": "SW",

        # Miscellaneous
        r"\bPOST OFFICE BOX\b": "PO BOX",
        r"\bP\.?\s*O\.?\s*BOX\b": "PO BOX",
    }

    # Apply replacements
    for pattern, replacement in replacements.items():
        series = series.str.replace(pattern, replacement, regex=True)

    # Remove spaces before commas
    series = series.str.replace(r"\s+,", ",", regex=True)

    # Ensure one space after commas
    series = series.str.replace(r",\s*", ", ", regex=True)

    # Remove duplicate commas
    series = series.str.replace(r",\s*,+", ",", regex=True)

    # Convert addresses containing only commas/spaces to blank
    series = series.replace(r"^[,\s]*$", "", regex=True)

    return series

# Import the excel data.
raw_df = pd.read_excel('2025 Roll - Parcels Export.xlsx', sheet_name='Only Delinquent Parcels')

## Filtering and cleaning the data
Eventually I will want to package this cleaning process into a function, so it can be used on future Kalamazoo County Excel files.

In [2]:
# Filtering down the original dataframe to only the necessary columns
df_original = raw_df[['ParcelMaster.parcelnumber', 'Combined Property Address', 'ParcelMaster.ownersname', 'ParcelMaster.ownersname2', 'ParcelMaster.ownercareof', 'Combined Owner Address', 'ParcelMaster.taxpayername', 'ParcelMaster.taxpayercareof', 'Combined Taxpayer Address', 'ParcelMaster.tax_send_to_name', 'Principal Residence Exemption %', 'Property Class', 'Taxable Value', 'Assessed Value', 'State Equalized Value (SEV)']]

# Renaming the orginal dataframes columns
df_original.columns = ['parcel_number', 'property_address', 'owners_name', 'owners_name_2', 'owner_care_of', 'owner_address',
              'taxpayer_name', 'taxpayer_care_of', 'taxpayer_address', 'tax_send_to_name', 'principal_residence_exemption_percentage',
              'property_class', 'tax_value', 'assessed_value', 'state_equalized_value']

# Create a new copy of the dataframe to be cleaned
df = df_original.copy()

# Define the entity columns.
# Entity columns are any columns where we might find an LLC, individual, or other organization/entity associated with a parcel.
entity_columns = ['owners_name', 'owners_name_2', 'owner_care_of', 'taxpayer_name', 'taxpayer_care_of', 'tax_send_to_name']

# Clean the entity columns
for col in entity_columns:
    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.upper()
        .str.strip()
    )

# Define the address columns.
address_columns = [
    "property_address",
    "owner_address",
    "taxpayer_address"
]

# Clean each of the address columns.
for col in address_columns:
    df[col] = normalize_address(df[col])

# Clean the property class column.
df["property_class"] = (
    df["property_class"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
)

## Performing initial analysis

In [3]:
df.shape
df.info()

print()
print('Checking for non-null values')
print()

(df != "").sum()

<class 'pandas.DataFrame'>
RangeIndex: 3874 entries, 0 to 3873
Data columns (total 15 columns):
 #   Column                                    Non-Null Count  Dtype
---  ------                                    --------------  -----
 0   parcel_number                             3874 non-null   str  
 1   property_address                          3874 non-null   str  
 2   owners_name                               3874 non-null   str  
 3   owners_name_2                             3874 non-null   str  
 4   owner_care_of                             3874 non-null   str  
 5   owner_address                             3874 non-null   str  
 6   taxpayer_name                             3874 non-null   str  
 7   taxpayer_care_of                          3874 non-null   str  
 8   taxpayer_address                          3874 non-null   str  
 9   tax_send_to_name                          3874 non-null   str  
 10  principal_residence_exemption_percentage  3874 non-null   int64
 11  pr

parcel_number                               3874
property_address                            3874
owners_name                                 3874
owners_name_2                                217
owner_care_of                                202
owner_address                               3874
taxpayer_name                               1979
taxpayer_care_of                             154
taxpayer_address                            1978
tax_send_to_name                            3844
principal_residence_exemption_percentage    3874
property_class                              3874
tax_value                                   3874
assessed_value                              3874
state_equalized_value                       3874
dtype: int64

### Observations
There are 3,874 total columns. This lines up with the number of populated rows in the Excel spreadsheet, meaning the data was imported successfully.

Entries that are not fully populated:
1. __217__ parcels have an `owners_name_2`.
2. __202__ parcels have an `owner_care_of`.
3. __1,979__ parcels (a little over half) have a `taxpayer_name`.
4. __154__ parcels have a `taxpayer_care_of`.
5. __3,844__ parcels (all except for 29) have a `tax_send_to_name`.

All other entries are fully populated.
- Notably, and thankfully, this means that all entries have a `taxpayer_address`.

Expressed another way, these are the amount of missing values/na values per column, listed from the most empty to the least empty...

In [4]:
(df == "").sum().sort_values(ascending=False)

taxpayer_care_of                            3720
owner_care_of                               3672
owners_name_2                               3657
taxpayer_address                            1896
taxpayer_name                               1895
tax_send_to_name                              30
owners_name                                    0
owner_address                                  0
parcel_number                                  0
property_address                               0
principal_residence_exemption_percentage       0
property_class                                 0
tax_value                                      0
assessed_value                                 0
state_equalized_value                          0
dtype: int64

and these are the amount of unique values per column...

In [5]:
df[['property_address', 'owners_name', 'owners_name_2', 'owner_care_of', 'owner_address', 'taxpayer_name', 'taxpayer_care_of', 'taxpayer_address', 'tax_send_to_name']].nunique().sort_values(ascending=True)

taxpayer_care_of     138
owner_care_of        185
owners_name_2        197
taxpayer_address    1559
taxpayer_name       1660
owner_address       3155
tax_send_to_name    3276
owners_name         3303
property_address    3778
dtype: int64

### Duplicates per column

By taking the number of non-empty cells for each column, and subtracting the number of unique values, we can find the number of cells which contain duplicates.

- `property_address`: 3874 - 3778 = **96 duplicates**
- `owners name`: 3874 - 3303 = **571 duplicates**
- `owners_name_2`: 217 - 196 = **21 duplicates**
- `owner_care_of`: 202 - 184 = **18 duplicates**
- `owner_address`: 3874 - 3155 = **719 duplicates**
- `taxpayer_name`: 1979 - 1660 = **319 duplicates**
- `taxpayer_care_of`: 154 - 184 = **17 duplicates**
- `taxpayer_address`: 3874 - 1566 = **2315 duplicates**
- `tax_send_to_name`: 3844 - 3276 = **568 duplicates**

## Deeper analysis

### LLCs

In [6]:
# Entity columns are any columns where we might find an LLC, individual, or other organization/entity associated with a parcel.
entity_columns = ['owners_name', 'owners_name_2', 'owner_care_of', 'taxpayer_name', 'taxpayer_care_of', 'tax_send_to_name']

# Creating a new dataframe which will contain only the rows in which an LLC is found in one or more of the entity columns.
llcs = df[
    df[entity_columns]
        .apply(lambda col: col.str.contains("LLC", case=False, na=False))
        .any(axis=1)
]

llcs.shape
llcs.info()

<class 'pandas.DataFrame'>
Index: 527 entries, 19 to 3863
Data columns (total 15 columns):
 #   Column                                    Non-Null Count  Dtype
---  ------                                    --------------  -----
 0   parcel_number                             527 non-null    str  
 1   property_address                          527 non-null    str  
 2   owners_name                               527 non-null    str  
 3   owners_name_2                             527 non-null    str  
 4   owner_care_of                             527 non-null    str  
 5   owner_address                             527 non-null    str  
 6   taxpayer_name                             527 non-null    str  
 7   taxpayer_care_of                          527 non-null    str  
 8   taxpayer_address                          527 non-null    str  
 9   tax_send_to_name                          527 non-null    str  
 10  principal_residence_exemption_percentage  527 non-null    int64
 11  propert

There are 527 rows which contain LLCs.

For all rows containing LLCs, there is no `owner_care_of` value provided.

LLC count by column
- `owners_name`: 493 LLCs
- `owners_name_2`: 10 LLCs
- `taxpayer_name`: 291 LLCs
- `taxpayer_care_of`: 2 LLCs
- `tax_send_to_name`: 488 LLCs

#### Number of unique LLCs

In [7]:
# Convert the entity-containing columns into a series, while removing missing values.
all_entities = (
    df[entity_columns]
    .stack()
    .dropna()
)

# Filter the series down to a list, consisting of only those entities which contain the string 'LLC.'
llcs_list = all_entities[
    all_entities.str.contains("LLC", case=False, na=False)
]

# Standardize the entity names
llcs_list = (
    llcs_list
    .str.upper()
    .str.strip()
)

# Create a list containing only unique LLCs (i.e. removing any duplicates)
unique_llcs = llcs_list.unique()

len(unique_llcs)

362

There are **362** unique LLCs in the dataset.

The list below shows the most frequently appearing LLCs (the count is how many times the LLCs appears in **any** field).

In [8]:
llcs_list.value_counts()

KAYLOR HOLDINGS LLC           62
STATE WIDE RENTALS LLC        38
19 PROPS, LLC                 21
WESTPOINTE DEVELOPMENT LLC    20
UJAMAA REALTY HOLDINGS LLC    18
                              ..
EVERCLARE LLC                  1
RSA KALAMAZOO LLC              1
KARNIMS PROPERTY ONE LLC       1
BAKERY LLC                     1
SWMAM LLC                      1
Name: count, Length: 362, dtype: int64

### Organizations

Rather than focusing solely on LLCs, I will now analyze the data to learn more about more organizations within the dataset, using a list of keywords used to identify these organizations.

In [9]:
import re

# The list below defines what words to search for to determine what counts as an organization.
organization_patterns = [
    r"\bLLC\b",
    r"\bINC\b",
    r"\bCORP\b",
    r"\bCORPORATION\b",
    r"\bCO\b",
    r"\bCOMPANY\b",
    r"\bLP\b",
    r"\bLTD\b",
    r"\bTRUST\b",
    r"\bBANK\b",
    r"\bCHURCH\b",
    r"\bCOUNTY\b",
    r"\bHOLDINGS\b",
    r"\bUNIVERSITY\b",
    r"\bTOWNSHIP\b",
    r"\bVILLAGE\b",
    r"\bDEPARTMENT\b",
    r"\bFOUNDATION\b",
    r"\bESTATE\b",
    r"\bESTATES\b"
]

# Create a regex search pattern.
pattern = "|".join(organization_patterns)

# Create a dataframe containing only the rows in which an organization is found
organizations_df = df[
    df[entity_columns]
        .apply(lambda col: col.str.contains(pattern, case=False, na=False, regex=True))
        .any(axis=1)
]

organizations_df.shape



(779, 15)

There are **779** rows containing organizations in total

#### Breakdown

In [10]:
# Turn the organizations dataframe into a MultiIndex list, containing every value from the dataframe.
# Note that parcel number and original column the organization was located in will be preserved.
organizations_list = (
    organizations_df[entity_columns]
    .stack()
    .dropna()
)

# Remove extraneous values (non-organization data such as addresses, etc. from the MultiIndex list).
organizations_list = (
    organizations_list[
        organizations_list.str.contains(pattern, case=False, na=False, regex=True)
    ]
    .str.upper()
    .str.strip()
    .drop_duplicates()
    .sort_values()
)

# Create a MultiIndex list that only contains unique instances of organizations
unique_orgs_list = organizations_list.unique()

# Get the total number of unique organizations
len(unique_orgs_list)

584

There are **584** unique organizations in the dataset.

- **362** LLC
- **126** TRUST
- **39** ESTATE
- **31** INC
- **24** HOLDINGS
- **13** CO
- **7** CORP
- **7** COMPANY
- **6** BANK
- **4** LP
- **4** CHURCH
- **3** CORPORATION
- **2** VILLAGE
- **2** DEPARTMENT
- **2** ESTATES
- **1** LTD
- **1** COUNTY
- **1** UNIVERSITY
- **1** TOWNSHIP
- **1** FOUNDATION

There are none of the following: ASSOCIATION, CITY, HOLDING, SCHOOL, DISTRICT, AUTHORITY, HOSPITAL, IRREVOCABLE TRUST

In [11]:
# Convert the MultiIndex list back into a dataframe.
org_info_df = (organizations_list.rename('organization_name').reset_index())

# Rename the dataframe columns.
org_info_df.columns = [
    "parcel_number",
    "source_column",
    "organization_name"
]

# Check and validate dataframe info and structure.
org_info_df.info()
org_info_df.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 584 entries, 0 to 583
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   parcel_number      584 non-null    int64
 1   source_column      584 non-null    str  
 2   organization_name  584 non-null    str  
dtypes: int64(1), str(2)
memory usage: 13.8 KB


,parcel_number,source_column,organization_name
0,1207,owners_name,"""E"" STREET, INC."
1,2566,owners_name_2,"& BEAUTY, LLC"
2,791,owners_name_2,& DEVELOPMENT LLC
3,707,owners_name,1119 JEFFERSON LLC
4,706,owners_name,1119 JEFFERSON LLC
5,1374,owners_name,125 KALAMAZOO MALL LLC
6,1093,owners_name,"1322 WASHINGTON, LLC"
7,420,owners_name,1560 S 8TH ST LLC
8,1611,owners_name,1733 W. MAIN ST LLC
9,783,owners_name,1808 KROM LLC


The `org_info_df` df created above will be helpful later on when creating entity tables.

### Cases where owner and taxpayer differ

In [12]:
different = df[
    (df["owners_name"] != "") &
    (df["taxpayer_name"] != "") &
    (df["owners_name"] != df["taxpayer_name"])
]

different.shape

(363, 15)

There are **363** cases where owner and taxpayer differ.

### Cases with different owners having the same taxpayer address

In [13]:
owner_counts = (
    df[df["taxpayer_address"] != ""]
    .groupby("taxpayer_address")["owners_name"]
    .nunique()
)

interesting_addresses = owner_counts[owner_counts > 1].index

# The results dataframe contains all rows with taxpayer addresses that are associated with more than one owner name.
results = (
    df[df["taxpayer_address"].isin(interesting_addresses)]
    .sort_values(["taxpayer_address", "owners_name"])
)

results

,parcel_number,property_address,owners_name,owners_name_2,owner_care_of,owner_address,taxpayer_name,taxpayer_care_of,taxpayer_address,tax_send_to_name,principal_residence_exemption_percentage,property_class,tax_value,assessed_value,state_equalized_value
1721,06-21-214-009,"824 NORMAL CT, KALAMAZOO, MI 49007","DOUBLE A PROPERTIES, LLC",,,"10017 TAPLIN LN, PLAINWELL, MI 49080","DOUBLE A PROPERTIES, LLC",,"10017 TAPLIN LN, PLAINWELL, MI 49080","DOUBLE A PROPERTIES, LLC",0,RESIDENTIAL VACANT,1759,2100,2100
1723,06-21-214-017,"826 NORMAL CT, KALAMAZOO, MI 49007","DOUBLE A PROPERTIES, LLC",,,"1090 12TH ST, MARTIN, MI 49070","DOUBLE A PROPERTIES, LLC",,"10017 TAPLIN LN, PLAINWELL, MI 49080","DOUBLE A PROPERTIES, LLC",0,RESIDENTIAL IMPROVED,24453,52900,52900
1724,06-21-214-020,"823 NORMAL CT, KALAMAZOO, MI 49007","GYORKOS, ANDREW",,,"10017 TAPLIN LN, PLAINWELL, MI 49080","GYORKOS, ANDREW",,"10017 TAPLIN LN, PLAINWELL, MI 49080","GYORKOS, ANDREW",0,RESIDENTIAL VACANT,2659,9800,9800
1269,06-15-115-011,"931 PRINCETON AVE, KALAMAZOO, MI 49007","LOVE, SHAQUILLE",,,"130 CANDLEWYCK DR APT 513, KALAMAZOO, MI 49001","LOVE, SHAQUILLE",,"1006 PRINCETON AVE, KALAMAZOO, MI 49007","LOVE, SHAQUILLE",0,RESIDENTIAL VACANT,3196,3600,3600
1270,06-15-116-001,"1006 PRINCETON AVE, KALAMAZOO, MI 49007","SUPER, KRISTIE",,"SUPER, CHARLES","1006 PRINCETON AVE, KALAMAZOO, MI 49007","SUPER, KRISTIE","SUPER, CHARLES","1006 PRINCETON AVE, KALAMAZOO, MI 49007","SUPER, KRISTIE",100,RESIDENTIAL IMPROVED,18383,36500,36500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
825,06-10-359-013,"1405 N ROSE ST, KALAMAZOO, MI 49007","ANDERSON, A DAVID SR & BERNADETTE",,,"PO BOX 50204, KALAMAZOO, MI 49005-0204","ANDERSON, A DAVID SR & BERNADETTE",,"PO BOX 50204, KALAMAZOO, MI 49005-0204","ANDERSON, A DAVID SR & BERNADETTE",0,RESIDENTIAL VACANT,928,4000,4000
838,06-10-364-001,"1357 N ROSE ST, KALAMAZOO, MI 49007","ANDERSON, A DAVID SR & BERNADETTE",,,"PO BOX 50204, KALAMAZOO, MI 49005-0204","ANDERSON, A DAVID SR & BERNADETTE",,"PO BOX 50204, KALAMAZOO, MI 49005-0204","ANDERSON, A DAVID SR & BERNADETTE",100,RESIDENTIAL IMPROVED,21775,42500,42500
840,06-10-364-014,"1401 N ROSE ST, KALAMAZOO, MI 49007","ANDERSON, A DAVID SR & BERNADETTE",,,"PO BOX 50204, KALAMAZOO, MI 49005-0204","ANDERSON, A DAVID SR & BERNADETTE",,"PO BOX 50204, KALAMAZOO, MI 49005-0204","ANDERSON, A DAVID SR & BERNADETTE",0,RESIDENTIAL VACANT,928,4000,4000
1289,06-15-143-017,"928 N EDWARDS ST, KALAMAZOO, MI 49007","ANDERSON, A DAVID SR & BERNADETTE",,,"PO BOX 50204, KALAMAZOO, MI 49005-0204","ANDERSON, A DAVID SR & BERNADETTE",,"PO BOX 50204, KALAMAZOO, MI 49005-0204","ANDERSON, A DAVID SR & BERNADETTE",0,RESIDENTIAL IMPROVED,11828,50800,50800


There are **384** rows containing a taxpayer address associated with more than one owner name.

For example, the taxpayer address 10017 TAPLIN LN, PLAINWELL, MI 49080 is used by DOUBLE A PROPERTIES, LLC for parcels 1721 and 1723, but is used by GYORKOS, ANDREW for parcel 1724.

In [14]:
# Produces a dataframe of all taxpayer addresses that are associated with more than one owner name
# and provides the number of associated unique owners, unique taxpayers, and unique parcels.
address_summary = (
    df[df["taxpayer_address"] != ""]
    .groupby("taxpayer_address")
    .agg(
        unique_owners=("owners_name", "nunique"),
        unique_taxpayers=("taxpayer_name", "nunique"),
        unique_parcels=("parcel_number", "nunique")
    )
    .query("unique_owners > 1")
    .sort_values("unique_owners", ascending=False)
)

address_summary

,unique_owners,unique_taxpayers,unique_parcels
taxpayer_address,,,
"8258 VICTORY, VICKSBURG, MI 49097",9,1,9
"200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 49007",7,7,17
"218 GLADYS ST, PORTAGE, MI 49002",5,5,7
"3001 10TH ST, WAYLAND, MI 49348",5,5,5
"12195 E MICHIGAN, GALESBURG, MI 49053",3,1,3
...,...,...,...
"PO BOX 2312, KALAMAZOO, MI 49003",2,2,4
"PO BOX 312, PORTAGE, MI 49081",2,2,5
"PO BOX 365, GALESBURG, MI 49053",2,2,2


## Key Findings

- Delinquent Parcels dataset contains 3874 rows/parcels.
    - Of these, 779 rows contain organizations, as defined by the `organization_patterns` variable.
- There are 3303 unique owner names.
- There are 1660 unique taxpayer names.
- 363 parcels have a taxpayer name that differs from the owner name.
- 124 taxpayer addresses are associated with multiple owner names.
- Data quality issues addressed:
    - 3 cells contained whitespace which stopped them from being identified as null values.
    - Address normalization revealed 43 addresses were identical to another address in the dataset, but were not shown as such due to the differences in wording used to input the addresses.